In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    KFold,
    StratifiedKFold,
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    RandomForestClassifier
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import joblib

In [2]:
reg_df = pd.read_csv(
    "../data/processed/regression_dataset.csv"
)

reg_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_estimated_delivery_date,delivery_days,customer_zip_code_prefix,customer_city,customer_state,...,seller_state,seller_zip_code_prefix,total_payment_value,number_of_payment_records,average_payment_value,max_installments,order_hour,order_day_of_week,order_month,is_weekend
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-18,8.436574,3149,sao paulo,SP,...,SP,9350,38.71,3.0,12.903333,1.0,10,0,10,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-08-13,13.782037,47813,barreiras,BA,...,SP,31570,141.46,1.0,141.460000,1.0,20,1,7,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-09-04,9.394213,75265,vianopolis,GO,...,SP,14840,179.12,1.0,179.120000,3.0,8,2,8,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-12-15,13.208750,59296,sao goncalo do amarante,RN,...,MG,31842,72.20,1.0,72.200000,1.0,19,5,11,1
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-26,2.873877,9195,santo andre,SP,...,SP,8752,28.62,1.0,28.620000,1.0,21,1,2,0


In [3]:
reg_df.shape

(96455, 36)

In [4]:
clf_df = pd.read_csv(
    "../data/processed/classification_dataset.csv"
)

clf_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_estimated_delivery_date,customer_zip_code_prefix,customer_city,customer_state,number_of_items,...,seller_zip_code_prefix,total_payment_value,number_of_payment_records,average_payment_value,max_installments,order_hour,order_day_of_week,order_month,is_weekend,late_delivery
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-18,3149,sao paulo,SP,1,...,9350,38.71,3.0,12.903333,1.0,10,0,10,0,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-08-13,47813,barreiras,BA,1,...,31570,141.46,1.0,141.460000,1.0,20,1,7,0,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-09-04,75265,vianopolis,GO,1,...,14840,179.12,1.0,179.120000,3.0,8,2,8,0,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-12-15,59296,sao goncalo do amarante,RN,1,...,31842,72.20,1.0,72.200000,1.0,19,5,11,1,0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-26,9195,santo andre,SP,1,...,8752,28.62,1.0,28.620000,1.0,21,1,2,0,0


In [5]:
X_reg = reg_df.drop(
    columns=["delivery_days"]
)

y_reg = reg_df["delivery_days"]

In [6]:
X_reg = X_reg.drop(
    columns=[
        "order_id",
        "customer_id"
    ],
    errors="ignore"
)

In [7]:
X_reg = X_reg.drop(
    columns=[
        "order_delivered_customer_date",
        "order_delivered_carrier_date",
        "delivery_delay_days",
        "late_delivery"
    ],
    errors="ignore"
)

In [8]:
X_reg = X_reg.drop(
    columns=[
        "order_purchase_timestamp",
        "order_estimated_delivery_date"
    ],
    errors="ignore"
)

In [9]:
numeric_features_reg = X_reg.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features_reg = X_reg.select_dtypes(
    include=["object"]
).columns.tolist()

C:\Users\acer\AppData\Local\Temp\ipykernel_10480\628359920.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features_reg = X_reg.select_dtypes(


In [10]:
numeric_pipeline_reg = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [11]:
categorical_pipeline_reg = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [12]:
preprocessor_reg = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline_reg,
            numeric_features_reg
        ),
        (
            "cat",
            categorical_pipeline_reg,
            categorical_features_reg
        )
    ]
)

In [13]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

In [14]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [15]:
rf_reg = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_reg
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=200,
                max_depth=15,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [16]:
cv_scores = cross_val_score(
    rf_reg,
    X_train_reg,
    y_train_reg,
    cv=kf,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

In [17]:
mae_scores = -cv_scores

In [18]:
print("Fold MAE:", mae_scores)
print("Mean MAE:", mae_scores.mean())
print("Std MAE:", mae_scores.std())

Fold MAE: [4.93456005 4.92824245 4.87899568 4.89887408 4.82841779]
Mean MAE: 4.893818010124743
Std MAE: 0.03838596273340275


In [19]:
param_grid_rf = {
    "model__n_estimators": [
        100,
        200,
        300
    ],

    "model__max_depth": [
        10,
        15,
        20,
        None
    ],

    "model__min_samples_split": [
        2,
        5,
        10
    ],

    "model__min_samples_leaf": [
        1,
        2,
        4
    ]
}

In [20]:
grid_search_reg = GridSearchCV(
    estimator=rf_reg,
    param_grid=param_grid_rf,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=1
)

In [ ]:
grid_search_reg.fit(
    X_train_reg,
    y_train_reg
)

Fitting 3 folds for each of 108 candidates, totalling 324 fits


In [ ]:
grid_search_reg.best_params_

In [ ]:
best_cv_mae = -grid_search_reg.best_score_

print(
    "Best CV MAE:",
    best_cv_mae
)

In [ ]:
best_reg_model = grid_search_reg.best_estimator_

In [ ]:
reg_pred = best_reg_model.predict(
    X_test_reg
)

In [ ]:
final_mae = mean_absolute_error(
    y_test_reg,
    reg_pred
)

In [ ]:
final_rmse = np.sqrt(
    mean_squared_error(
        y_test_reg,
        reg_pred
    )
)

In [ ]:
final_r2 = r2_score(
    y_test_reg,
    reg_pred
)

In [ ]:
print("Final MAE :", final_mae)
print("Final RMSE:", final_rmse)
print("Final R²  :", final_r2)

In [ ]:
X_clf = clf_df.drop(
    columns=["late_delivery"]
)

y_clf = clf_df["late_delivery"]

In [ ]:
X_clf = X_clf.drop(
    columns=[
        "order_id",
        "customer_id"
    ],
    errors="ignore"
)

In [ ]:
X_clf = X_clf.drop(
    columns=[
        "order_delivered_customer_date",
        "order_delivered_carrier_date",
        "delivery_delay_days",
        "delivery_days"
    ],
    errors="ignore"
)

In [ ]:
X_clf = X_clf.drop(
    columns=[
        "order_purchase_timestamp",
        "order_estimated_delivery_date"
    ],
    errors="ignore"
)

In [ ]:
numeric_features_clf = X_clf.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features_clf = X_clf.select_dtypes(
    include=["object"]
).columns.tolist()

In [ ]:
numeric_pipeline_clf = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [ ]:
categorical_pipeline_clf = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [ ]:
preprocessor_clf = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline_clf,
            numeric_features_clf
        ),
        (
            "cat",
            categorical_pipeline_clf,
            categorical_features_clf
        )
    ]
)

In [ ]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf,
    y_clf,
    test_size=0.2,
    random_state=42,
    stratify=y_clf
)

In [ ]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
rf_clf = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_clf
        ),
        (
            "model",
            RandomForestClassifier(
                random_state=42,
                n_jobs=-1,
                class_weight="balanced"
            )
        )
    ]
)

In [ ]:
cv_f1 = cross_val_score(
    rf_clf,
    X_train_clf,
    y_train_clf,
    cv=skf,
    scoring="f1",
    n_jobs=-1
)

In [ ]:
print("F1 scores:", cv_f1)
print("Mean F1:", cv_f1.mean())
print("Std F1:", cv_f1.std())

In [ ]:
cv_auc = cross_val_score(
    rf_clf,
    X_train_clf,
    y_train_clf,
    cv=skf,
    scoring="roc_auc",
    n_jobs=-1
)

In [ ]:
print("ROC-AUC scores:", cv_auc)
print("Mean ROC-AUC:", cv_auc.mean())

In [ ]:
param_grid_clf = {
    "model__n_estimators": [
        100,
        200,
        300
    ],

    "model__max_depth": [
        10,
        15,
        20,
        None
    ],

    "model__min_samples_split": [
        2,
        5,
        10
    ],

    "model__min_samples_leaf": [
        1,
        2,
        4
    ],

    "model__max_features": [
        "sqrt",
        "log2"
    ]
}

In [ ]:
random_search_clf = RandomizedSearchCV(
    estimator=rf_clf,
    param_distributions=param_grid_clf,
    n_iter=20,
    cv=3,
    scoring="f1",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [ ]:
random_search_clf.fit(
    X_train_clf,
    y_train_clf
)

In [ ]:
random_search_clf.best_params_

In [ ]:
print(
    "Best CV F1:",
    random_search_clf.best_score_
)

In [ ]:
best_clf_model = (
    random_search_clf.best_estimator_
)

In [ ]:
clf_pred = best_clf_model.predict(
    X_test_clf
)

In [ ]:
clf_prob = best_clf_model.predict_proba(
    X_test_clf
)[:, 1]

In [ ]:
final_accuracy = accuracy_score(
    y_test_clf,
    clf_pred
)

final_precision = precision_score(
    y_test_clf,
    clf_pred,
    zero_division=0
)

final_recall = recall_score(
    y_test_clf,
    clf_pred,
    zero_division=0
)

final_f1 = f1_score(
    y_test_clf,
    clf_pred,
    zero_division=0
)

final_auc = roc_auc_score(
    y_test_clf,
    clf_prob
)

In [ ]:
print("Accuracy :", final_accuracy)
print("Precision:", final_precision)
print("Recall   :", final_recall)
print("F1       :", final_f1)
print("ROC-AUC  :", final_auc)

In [ ]:
cm = confusion_matrix(
    y_test_clf,
    clf_pred
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "On Time",
        "Late"
    ]
)

disp.plot()

plt.title(
    "Final Random Forest Confusion Matrix"
)

plt.show()

In [ ]:
clf_prob >= 0.5

In [ ]:
thresholds = [
    0.3,
    0.4,
    0.5,
    0.6,
    0.7
]

In [ ]:
threshold_results = []

for threshold in thresholds:

    threshold_pred = (
        clf_prob >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": threshold,

        "Precision": precision_score(
            y_test_clf,
            threshold_pred,
            zero_division=0
        ),

        "Recall": recall_score(
            y_test_clf,
            threshold_pred,
            zero_division=0
        ),

        "F1": f1_score(
            y_test_clf,
            threshold_pred,
            zero_division=0
        )
    })

In [ ]:
threshold_df = pd.DataFrame(
    threshold_results
)

threshold_df

In [ ]:
best_threshold_row = threshold_df.loc[
    threshold_df["F1"].idxmax()
]

best_threshold_row

In [ ]:
best_threshold_row = threshold_df.loc[
    threshold_df["F1"].idxmax()
]

best_threshold_row

In [ ]:
final_threshold_pred = (
    clf_prob >= best_threshold
).astype(int)

In [ ]:
print(
    "Precision:",
    precision_score(
        y_test_clf,
        final_threshold_pred,
        zero_division=0
    )
)

print(
    "Recall:",
    recall_score(
        y_test_clf,
        final_threshold_pred,
        zero_division=0
    )
)

print(
    "F1:",
    f1_score(
        y_test_clf,
        final_threshold_pred,
        zero_division=0
    )
)

In [ ]:
joblib.dump(
    best_reg_model,
    "../models/final_delivery_time_model.pkl"
)

In [ ]:
joblib.dump(
    best_clf_model,
    "../models/final_late_delivery_model.pkl"
)

In [ ]:
joblib.dump(
    best_threshold,
    "../models/late_delivery_threshold.pkl"
)